# Notebook 08 — Leakage Control and Split Design

## Purpose
I define the train/test split and explicitly check for data leakage before
any model is trained.

## Why this matters
Leakage causes over-optimistic results that do not hold in production.
Checking it explicitly, in a dedicated notebook, makes the decision visible
and reproducible.

## The split strategy
I use a **time-based split** rather than random split because:
1. Orders are time-ordered events (not IID samples).
2. A random split can put future events in the training set, leaking signal.
3. Time-based splits simulate real deployment: train on past, predict future.

Split date: 2018-01-01 (defined in configs/config.yaml)


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path('..').resolve()))
from src.config import load_config
from src.paths import Paths
from src.utils import set_all_seeds

cfg   = load_config()
paths = Paths(cfg)
SEED  = cfg['project']['random_seed']
set_all_seeds(SEED)

master = pd.read_parquet(paths.processed / cfg['data']['master_file'])
print(f"Master loaded: {master.shape}")


  Random seed set to 42
Master loaded: (96478, 33)


In [2]:
# Time-based split
SPLIT_DATE = pd.Timestamp(cfg['modeling']['time_split_date'])
print(f"Split date: {SPLIT_DATE.date()}")

train_mask = master['order_purchase_timestamp'] < SPLIT_DATE
test_mask  = master['order_purchase_timestamp'] >= SPLIT_DATE

train = master[train_mask].copy().reset_index(drop=True)
test  = master[test_mask].copy().reset_index(drop=True)

print(f"Train: {len(train):,} orders "
      f"({train['order_purchase_timestamp'].min().date()} – "
      f"{train['order_purchase_timestamp'].max().date()})")
print(f"Test:  {len(test):,} orders "
      f"({test['order_purchase_timestamp'].min().date()} – "
      f"{test['order_purchase_timestamp'].max().date()})")
print(f"Train/test ratio: {len(train)/len(master)*100:.1f}% / {len(test)/len(master)*100:.1f}%")


Split date: 2018-01-01
Train: 43,695 orders (2016-09-15 – 2017-12-31)
Test:  52,783 orders (2018-01-01 – 2018-08-29)
Train/test ratio: 45.3% / 54.7%


In [3]:
# Leakage check 1: target label distribution in train vs test
print("=== Target Distribution Check ===")
for target in ['review_score', 'is_late']:
    if target in master.columns:
        train_dist = train[target].dropna().value_counts(normalize=True).sort_index()
        test_dist  = test[target].dropna().value_counts(normalize=True).sort_index()
        print(f"\n{target}:")
        print(pd.DataFrame({'train': train_dist, 'test': test_dist}).round(3))


=== Target Distribution Check ===

review_score:
              train   test
review_score              
1.0           0.090  0.104
2.0           0.031  0.030
3.0           0.085  0.080
4.0           0.203  0.193
5.0           0.591  0.593

is_late:
         train   test
is_late              
0.0      0.944  0.923
1.0      0.056  0.077


In [4]:
# Leakage check 2: No customer appears in BOTH train and test (time split handles this)
train_customers = set(train['customer_unique_id'])
test_customers  = set(test['customer_unique_id'])
overlap = train_customers & test_customers

print(f"Customers in train: {len(train_customers):,}")
print(f"Customers in test:  {len(test_customers):,}")
print(f"Customer overlap:   {len(overlap):,}")
print()
if len(overlap) > 0:
    print(f"NOTE: {len(overlap)} customers appear in both sets.")
    print("This is expected — a customer who purchased in 2017 may also purchase in 2018.")
    print("The split is order-level, not customer-level, which is correct for this task.")
    print("We are NOT leaking future purchase info into training.")


Customers in train: 42,395
Customers in test:  51,612
Customer overlap:   649

NOTE: 649 customers appear in both sets.
This is expected — a customer who purchased in 2017 may also purchase in 2018.
The split is order-level, not customer-level, which is correct for this task.
We are NOT leaking future purchase info into training.


In [5]:
# Leakage check 3: confirm delivery_delay_days is NOT used in late-delivery prediction
FEAT_LATE = cfg['modeling']['feature_cols_late']
FEAT_REVIEW = cfg['modeling']['feature_cols_review']

print("Features for LATE DELIVERY prediction (Task A):")
for f in FEAT_LATE:
    print(f"  {f}")
print()
assert 'delivery_delay_days' not in FEAT_LATE, "LEAKAGE: delivery_delay_days is in late features!"
assert 'is_late' not in FEAT_LATE, "LEAKAGE: is_late is in its own features!"
print("LEAKAGE CHECK PASSED: delivery_delay_days is NOT in late-delivery features.")


Features for LATE DELIVERY prediction (Task A):
  order_value
  freight_value
  freight_ratio
  n_items
  product_weight_g
  purchase_month
  purchase_dayofweek
  seller_state_encoded
  customer_state_encoded
  product_category_encoded

LEAKAGE CHECK PASSED: delivery_delay_days is NOT in late-delivery features.


In [6]:
# Leakage check 4: scaling must be fit ONLY on train, not on full dataset
# I demonstrate this here — actual scaling happens inside model pipelines
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
# CORRECT: fit on train only
scaler.fit(train[['order_value']].fillna(0))
train_scaled = scaler.transform(train[['order_value']].fillna(0))
test_scaled  = scaler.transform(test[['order_value']].fillna(0))

print("Scaling check:")
print(f"  Scaler mean (from train): {scaler.mean_[0]:.2f}")
print(f"  Train order_value mean (raw): {train['order_value'].mean():.2f}")
print()
print("CORRECT: scaler is fit on train only, then applied to test.")


Scaling check:
  Scaler mean (from train): 137.39
  Train order_value mean (raw): 137.39

CORRECT: scaler is fit on train only, then applied to test.


In [7]:
# Save train/test indices for use in modelling notebooks
paths.processed.mkdir(parents=True, exist_ok=True)
train.to_parquet(paths.processed / 'train.parquet', index=False)
test.to_parquet( paths.processed / 'test.parquet',  index=False)
print(f"Saved train.parquet ({len(train):,} rows) and test.parquet ({len(test):,} rows)")
print()
print("Notebook 08 complete.")
print("Next: Notebook 09 — Baseline Models")


Saved train.parquet (43,695 rows) and test.parquet (52,783 rows)

Notebook 08 complete.
Next: Notebook 09 — Baseline Models
